<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/03d_visualizations_survival_rate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Analysis still in progress

##Importing

In [15]:

import pandas as pd
import geopandas as gpd
import numpy as np
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString
import plotly.express as px



from google.colab import drive
drive.mount('/content/drive')

!wget https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/functions.py
import functions as fx



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--2026-04-16 21:44:09--  https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8682 (8.5K) [text/plain]
Saving to: ‘functions.py.1’

functions.py.1      100%[===================>]   8.48K  --.-KB/s    in 0s      

2026-04-16 21:44:09 (81.8 MB/s) - ‘functions.py.1’ saved [8682/8682]



In [16]:
import functions
import importlib
importlib.reload(functions)

<module 'functions' from '/content/functions.py'>

##Reading in file

In [17]:
gdf = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/biz_all_startdate.geojson")

##Reading in Census Tracts

In [18]:
sf_tracts = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/sf_tracts.geojson")

In [25]:
sf_block_grp = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/sf_block_grp.geojson")
sf_block_grp = sf_block_grp.rename(columns={'geoid': 'GEOID'})
sf_block_grp.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [20]:
gdf

,uniqueid,business_account_number,location_id,ownership_name,dba_name,business_start_date,business_end_date,location_start_date,location_end_date,naics_code,...,lic_code_description,business_corridor,neighborhoods_analysis_boundaries,administratively_closed_bool,status,year_open,year_closed,lon,lat,geometry
0,1274046-04-211-1123120,1123120,1274046-04-211,Deans & Homer,Deans & Homer,1856-01-01,NaT,1856-01-01,NaT,5240-5249,...,None,None,Financial District/South Beach,False,closed,1856,NaN,-122.399428,37.792530,POINT (-122.39943 37.79253)
1,1094538-08-161-1045057,1045057,1094538-08-161,Congregation Sherith Israel,Congregation Sherith Israel,1870-02-17,NaT,1870-02-17,NaT,None,...,FOOD PREPARATION & SERVICE ESTABLISHMENT EXEMPT,None,Pacific Heights,False,closed,1870,NaN,-122.431824,37.789470,POINT (-122.43182 37.78947)
2,1335477-06-231-1147684,1147684,1335477-06-231,International Alliance Of Theatrical Stage Emp...,International Alliance Of Theatrical Stage Emp...,1893-05-01,NaT,1893-05-01,NaT,7100-7199,...,None,None,Financial District/South Beach,False,closed,1893,NaN,-122.397813,37.785942,POINT (-122.39781 37.78594)
3,1124942-10-161-1060029,1060029,1124942-10-161,Benevolent And Protective Order Elks San Franc...,San Francisco Elk's Lodge #3,1896-08-24,NaT,1896-08-24,NaT,None,...,Multiple,None,Nob Hill,False,closed,1896,NaN,-122.409383,37.788462,POINT (-122.40938 37.78846)
4,1176607-01-181-1060029,1060029,1176607-01-181,Benevolent And Protective Order Elks San Franc...,Elks Club,1896-08-24,NaT,1896-08-24,NaT,None,...,"RESTAURANT - OVER 2,000 SQFT",None,Nob Hill,False,closed,1896,NaN,-122.409383,37.788462,POINT (-122.40938 37.78846)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133040,1412728-02-261-1134074,1134074,1412728-02-261,Gonzalo Guoron Tzian,Kaqchikel Translations,2021-06-01,NaT,2026-02-01,NaT,8100-8139,...,None,None,Bayview Hunters Point,False,closed,2026,NaN,-122.387990,37.732968,POINT (-122.38799 37.73297)
133041,1413901-03-261-1159782,1159782,1413901-03-261,Hilary Grubb,"Hilary Maia Grubb, Md",2024-06-25,NaT,2026-03-02,NaT,6100-6299,...,None,None,Castro/Upper Market,False,closed,2026,NaN,-122.429457,37.768860,POINT (-122.42946 37.76886)
133042,1408790-01-261-1150296,1150296,1408790-01-261,Barset & Co LLC,Kalamari,2023-08-30,NaT,2026-01-22,NaT,5400-5499,...,None,None,Nob Hill,False,closed,2026,NaN,-122.421145,37.790217,POINT (-122.42115 37.79022)
133043,1410157-02-261-1154031,1154031,1410157-02-261,2525 Van Ness Owners Association,2525 Van Ness Owners Association,2024-01-16,NaT,2026-01-01,NaT,None,...,ERRCS,None,Marina,False,closed,2026,NaN,-122.424183,37.798800,POINT (-122.42418 37.7988)


In [21]:
pre_covid_mask = gdf['location_start_date'] < '2020-03-01'
pre_covid_tracts = gdf[pre_covid_mask]

survival_mask = pre_covid_tracts['location_end_date'].notna()
survival_tracts = pre_covid_tracts[survival_mask].sort_values('location_start_date')

In [22]:
pre_covid_tracts_grouped = functions.group_points_by_poly(
    points=pre_covid_tracts,
    polygons=sf_block_grp
)

survival_tracts_grouped = functions.group_points_by_poly(
    points=survival_tracts,
    polygons=sf_block_grp
)


# adding closed businesses and opened businesses that have not closed
# (because closed businesses also are listed in opened, just in a diff year)
survival_tracts_grouped['total'] = pre_covid_tracts_grouped['closed'] + survival_tracts_grouped['opened']
survival_tracts_grouped['survival_rate'] = survival_tracts_grouped['opened'] / survival_tracts_grouped['total']

survival_tracts_grouped

,GEOID,geometry,opened,total,survival_rate
0,60750119012,"MULTIPOLYGON (((-122.41411 37.79142, -122.4124...",25.0,45.0,0.555556
1,60750122021,"MULTIPOLYGON (((-122.41993 37.78683, -122.4182...",43.0,72.0,0.597222
2,60750135003,"MULTIPOLYGON (((-122.43585 37.7905, -122.4342 ...",100.0,201.0,0.497512
3,60750130022,"MULTIPOLYGON (((-122.43733 37.79781, -122.4356...",156.0,287.0,0.543554
4,60750168022,"MULTIPOLYGON (((-122.43082 37.77397, -122.4291...",35.0,66.0,0.530303
...,...,...,...,...,...
676,60750227042,"MULTIPOLYGON (((-122.40649 37.76069, -122.4064...",55.0,94.0,0.585106
677,60750204022,"MULTIPOLYGON (((-122.44968 37.74982, -122.4495...",36.0,82.0,0.439024
678,60750204021,"MULTIPOLYGON (((-122.45087 37.74608, -122.4507...",24.0,85.0,0.282353
679,60750204012,"MULTIPOLYGON (((-122.44792 37.75759, -122.4477...",24.0,53.0,0.452830


In [28]:
import plotly.graph_objects as go

fig = px.choropleth_mapbox(
    survival_tracts_grouped,
    geojson=survival_tracts_grouped.set_index("GEOID").__geo_interface__,
    locations="GEOID",
    color="survival_rate",
    hover_name="GEOID",
    hover_data={'opened':True},
    center={"lat": 37.7749, "lon": -122.4194},
    zoom=10,
    mapbox_style="carto-positron",
    color_continuous_scale="Reds",
    height=500,
    width=700,
    opacity=.1

)


fig.show()

In [ ]:
fig = px.density_mapbox(
    survival_tracts,
    lat=survival_tracts.geometry.y,
    lon=survival_tracts.geometry.x,
    radius=3,
    center={"lat": 37.7749, "lon": -122.4194},
    zoom=11,
    mapbox_style="carto-positron",
    height=600
)
fig.show()